# **FINAL SYSTEM**

In [2]:
import sys

#!pip install facenet-pytorch

src_path = "/content/drive/MyDrive/11/Consegna finale/src"

if src_path not in sys.path:
    sys.path.append(src_path)

import cv2
import numpy as np
import time
from biometric_system_copy import IntegratedBiometricSystem
from system_config import Config
from base64 import b64decode
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow

## **Sistema Biometrico Integrato (Main Pipeline)**

Questa sezione rappresenta il **core dell'applicazione**. La classe `IntegratedBiometricSystem` funge da orchestratore, unificando tutti i moduli sviluppati in precedenza (Acquisizione, Liveness, Feature Extraction) in un'unica pipeline operativa.



**Flusso di Esecuzione:**
1.  **Inizializzazione:** Carica le configurazioni, i pesi dei modelli (FaceNet + Liveness) e la Gallery esistente.
2.  **Acquisizione:** Utilizza la webcam (via JavaScript) per catturare il volto, filtrandolo attraverso i controlli di qualità e posa della classe `BiometricAcquisition`.
3.  **Liveness Check:** Se attivato, verifica che il volto sia reale (Anti-Spoofing) prima di procedere.
4.  **Modalità Operative:**
    * **Enrollment:** Registra un nuovo utente salvando il suo embedding su disco.
    * **Identification (1:N):** Cerca il volto in tutta la gallery (Open-Set) usando la soglia **T_i**.
    * **Verification (1:1):** Verifica l'identità rispetto a un ID dichiarato usando la soglia **T_v**.
5.  **Monitoraggio:** Stampa a video i tempi di esecuzione e l'occupazione di memoria (RAM/VRAM) per verificare il rispetto dei vincoli hardware.

In [ ]:
def capture_image_colab(quality=0.8):
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Cattura Foto';
        div.appendChild(capture);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
      ''')
    display(js)
    try:
        data = eval_js('takePhoto({})'.format(quality))
        if not data: return None
        binary = b64decode(data.split(',')[1])
        return cv2.imdecode(np.frombuffer(binary, np.uint8), -1)
    except Exception: return None


cfg = Config()
bio_system = IntegratedBiometricSystem(cfg)
bio_system.run_pipeline(capture_func=capture_image_colab)

>>> Inizializzazione Sistema Biometrico...
[LIB] Detector inizializzato su cpu. Controllo Yaw + Pitch attivo.
Inizializzazione Modello su cpu...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.

Caricamento Gallery da file: /content/drive/MyDrive/11/Consegna parziale/features_labels_gr_11 e /content/drive/MyDrive/11/Consegna parziale/features_embeddings_gr_11...
Gallery caricata: 160 soggetti.
--- Caricamento pesi Liveness da: /content/drive/MyDrive/11/Consegna parziale/models/best_weights_spoof_gr_11.pth ---
   >>> Rilevato CheckpointManager (Best Loss: 0.08074173726004119)
>>> Modello Liveness pronto.

--- SELEZIONA MODALITÀ ---
1. Enrollment (Registra nuovo utente)
2. Identification (Chi sono? - Open Set)
3. Verification (Io sono l'utente X, verificami)
Inserisci numero (1/2/3): 2
Attivare Liveness Detection? (s/n): s

>>> Avvio Acquisizione (Guardare in camera)...
Inizio procedura (Yaw Th: 0.3, Pitch Range: (0.35, 1.8))...

--- Tentativo 1/3 ---


<IPython.core.display.Javascript object>

Volto FRONTALE: B=237, C=64
Qualità e Posa 3D accettate!
>>> Acquisizione completata. Avvio timer elaborazione...
>>> Eseguo controllo Liveness...
   -> Liveness Score: 0.9800 (LIVE)

RISULTATO IDENTIFICATION (Soglia Ti=0.7898):
Trovati 8 possibili match:
   1. Utente 2 (Distanza: 0.5931)
   2. Utente 2 (Distanza: 0.6157)
   3. Utente 2 (Distanza: 0.6739)
   4. Utente 2 (Distanza: 0.6746)
   5. Utente 2 (Distanza: 0.6946)
   6. Utente 2 (Distanza: 0.7018)
   7. Utente 2 (Distanza: 0.7413)
   8. Utente 2 (Distanza: 0.7512)
>>> IDENTIFICATO COME: Utente 2

Tempo Elaborazione (Post-Acquisizione): 0.26 secondi.

--- STATISTICHE MEMORIA ---
RAM Utilizzata (Processo): 1.14 GB
Totale Stimato: 1.14 GB
>>> Requisito Memoria (< 4GB) RISPETTATO.


In [ ]:
cfg = Config()
bio_system = IntegratedBiometricSystem(cfg)
folder_path = "/content/drive/MyDrive/foto_test/gallery/subject_20"
bio_system.batch_enroll_from_folder(folder_path, 20)

>>> Inizializzazione Sistema Biometrico...
[LIB] Detector inizializzato su cpu. Controllo Yaw + Pitch attivo.
Inizializzazione Modello su cpu...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.

Caricamento Gallery da file: /content/drive/MyDrive/11/Consegna parziale/features_labels_gr_11_copy e /content/drive/MyDrive/11/Consegna parziale/features_embeddings_gr_11_copy...
Gallery caricata: 160 soggetti.
--- Caricamento pesi Liveness da: /content/drive/MyDrive/11/Consegna parziale/models/best_weights_spoof_gr_11.pth ---
   >>> Rilevato CheckpointManager (Best Loss: 0.08074173726004119)
>>> Modello Liveness pronto.

>>> AVVIO BATCH ENROLLMENT per Utente ID: 20
   Cartella: /content/drive/MyDrive/foto_test/gallery/subject_20 (3 files)
   - gallery_01.jpeg: OK. Estrazione features... Fatto.
   - gallery_02.jpeg: OK. Estrazione features... Fatto.
   - gallery_03.jpeg: OK. Estrazione features... Fatto.

>>> Aggiunta di 3 vettori alla Gallery...
>>> Gallery Aggiornata e 

## **Sistema Biometrico Integrato (Main Pipeline)**

Questa sezione rappresenta il **core dell'applicazione**. La classe `IntegratedBiometricSystem` funge da orchestratore, unificando tutti i moduli sviluppati in precedenza (Acquisizione, Liveness, Feature Extraction) in un'unica pipeline operativa flessibile e modulare.

**Flusso di Esecuzione:**
1.  **Inizializzazione:** Carica le configurazioni di sistema, i pesi dei modelli (FaceNet per il riconoscimento, Modello Custom per il Liveness) e la Gallery degli utenti registrati (file `.npy`).
2.  **Acquisizione Ibrida:** Gestisce l'input biometrico da due sorgenti:
    * **Live (Webcam):** Cattura in tempo reale con logica di *retries* automatica.
    * **Test (Folder):** Elabora immagini statiche da una directory per test massivi e debug.
    In entrambi i casi, l'immagine passa attraverso il **Quality Gate** della classe `BiometricAcquisition` (controllo sfocatura, contrasto e posa 3D Yaw/Pitch).
3.  **Liveness Check (Opzionale):** Se attivato dall'utente, il sistema interroga il modello Anti-Spoofing per classificare il volto come *Real* o *Fake*, bloccando tentativi di attacco prima della fase di riconoscimento.
4.  **Modalità Operative:**
    * **Enrollment:** Estrae le feature e registra un nuovo utente salvando il vettore nella Gallery.
    * **Identification (1:N):** Confronta il volto con l'intera Gallery (Open-Set) utilizzando la soglia di distanza **$T_i$**.
    * **Verification (1:1):** Verifica l'identità rispetto a un ID dichiarato confrontando 1:1 con la soglia **$T_v$**.
5.  **Monitoraggio:** Fornisce feedback in tempo reale su tempi di latenza e utilizzo risorse (RAM/VRAM), garantendo il rispetto dei vincoli hardware del progetto.

In [3]:
config = Config()
system = IntegratedBiometricSystem(config)

cartella_test = "/content/drive/MyDrive/foto_test/probe/subject_20"
system.run_folder_testing(cartella_test)

>>> Inizializzazione Sistema Biometrico...
[LIB] Detector inizializzato su cuda. Controllo Yaw + Pitch attivo.
Inizializzazione Modello su cuda...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.


  0%|          | 0.00/107M [00:00<?, ?B/s]


Caricamento Gallery da file: /content/drive/MyDrive/11/Consegna parziale/features_labels_gr_11_copy e /content/drive/MyDrive/11/Consegna parziale/features_embeddings_gr_11_copy...
Gallery caricata: 163 soggetti.
--- Caricamento pesi Liveness da: /content/drive/MyDrive/11/Consegna parziale/models/best_weights_spoof_gr_11.pth ---
   >>> Rilevato CheckpointManager (Best Loss: 0.08074173726004119)
>>> Modello Liveness pronto.
Attivare Liveness Detection per questo test? (s/n): n

>>> AVVIO TEST SU CARTELLA: /content/drive/MyDrive/foto_test/probe/subject_20 (3 files)
>>> Modalità Liveness: DISATTIVATA

FILE [1/3]: probe_01.jpeg
Report Acquisizione:
   Volto GIRATO(score 0.42): B=115, C=43 -> Esito: RIFIUTATO (Testa girata orizzontalmente (0.42))
!!! FALLIMENTO QUALITY GATE !!!
Vuoi forzare il test comunque? (s/n): s
>>> FORZATURA ESEGUITA.
>>> Liveness Check: DISATTIVATO (Skipped)

Immagine pronta: probe_01.jpeg
1. Identification (Chi è?)
2. Verification (Verifica ID specifico)
3. Skip (Pa

In [4]:
config = Config()
system = IntegratedBiometricSystem(config)

cartella_test = "/content/drive/MyDrive/foto_test/probe/subject_20"
system.run_folder_testing(cartella_test)

>>> Inizializzazione Sistema Biometrico...
[LIB] Detector inizializzato su cuda. Controllo Yaw + Pitch attivo.
Inizializzazione Modello su cuda...
ATTENZIONE: Nessun peso custom trovato/fornito. Uso VGGFace2 standard.

Caricamento Gallery da file: /content/drive/MyDrive/11/Consegna parziale/features_labels_gr_11_copy e /content/drive/MyDrive/11/Consegna parziale/features_embeddings_gr_11_copy...
Gallery caricata: 163 soggetti.
--- Caricamento pesi Liveness da: /content/drive/MyDrive/11/Consegna parziale/models/best_weights_spoof_gr_11.pth ---
   >>> Rilevato CheckpointManager (Best Loss: 0.08074173726004119)
>>> Modello Liveness pronto.
Attivare Liveness Detection per questo test? (s/n): s

>>> AVVIO TEST SU CARTELLA: /content/drive/MyDrive/foto_test/probe/subject_20 (3 files)
>>> Modalità Liveness: ATTIVA

FILE [1/3]: probe_01.jpeg
Report Acquisizione:
   Volto GIRATO(score 0.42): B=115, C=43 -> Esito: RIFIUTATO (Testa girata orizzontalmente (0.42))
!!! FALLIMENTO QUALITY GATE !!!
Vuo